# WS2 Thermal CVD — Bayesian Optimization
**Objective:** Minimize PL FWHM of WS2 films using Gaussian Process Regression + Expected Improvement.

Uses the same ML pipeline as the production app backend:
- Dataset: `WS2_ThermalCVD_BlankCells_17points.xlsx`
- Kernel: ARD Matern nu=2.5 + WhiteKernel (`gp_model.py`)
- Preprocessing: StandardScaler + SimpleImputer (`data_encoder.py`)
- Search: Latin Hypercube 5000 pts (`optimizer.py`)

## Step 0 — Install & Import

In [ ]:
!pip install scikit-learn scipy numpy pandas openpyxl matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 150, 'font.family': 'DejaVu Sans',
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Libraries loaded.')

## Step 1 — Load Dataset
Loads `WS2_ThermalCVD_BlankCells_17points.xlsx` — the same file used by the app.

In [ ]:
# Same preprocessing as thermal_cvd_routes.py
FILE = 'WS2_ThermalCVD_BlankCells_17points.xlsx'
df_raw = pd.read_excel(FILE)

# Replace 'NS' with NaN — matches app logic
df_raw = df_raw.replace('NS', np.nan)

# Normalise PL FWHM column name
if 'PL_FWHM' not in df_raw.columns and 'PL FWHM' in df_raw.columns:
    df_raw = df_raw.rename(columns={'PL FWHM': 'PL_FWHM'})

print(f'File: {FILE}')
print(f'Rows: {len(df_raw)}  |  Columns: {len(df_raw.columns)}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

## Step 2 — Filter Thermal CVD
Keep only rows where `TOCVD == 'Thermal CVD'` — identical to the app filter.

In [ ]:
# Matches: df[df['TOCVD'] == 'Thermal CVD'] in data_encoder.py
df = df_raw[df_raw['TOCVD'] == 'Thermal CVD'].copy().reset_index(drop=True)

# Coerce numeric columns
num_cols = ['FRH', 'HR', 'FRP1', 'FRP2', 'CP1', 'CP2',
            'GTE', 'GTI', 'FRA', 'Pressure', 'PL_FWHM']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Thermal CVD rows found: {len(df)}')
print(f'FWHM range: {df["PL_FWHM"].min()} to {df["PL_FWHM"].max()} meV')
display(df.style.set_caption('Thermal CVD Dataset').background_gradient(
    subset=['PL_FWHM'], cmap='RdYlGn_r'))

## Step 3 — Constants & Optimization Variables
Exact same classification as `ThermalCVDEncoder` in `data_encoder.py`.

In [ ]:
# From app/ml_models/thermal_cvd/data_encoder.py

# Categorical constants (fixed per experiment setup)
CAT_CONSTANTS = ['P1', 'P2', 'Substrate', 'CG', 'COM', 'PC', 'SA', 'Class']

# Numerical constants (not optimised)
NUM_CONSTANTS = ['FRH', 'HR', 'FRP1', 'FRP2', 'CP1', 'CP2']

# The 4 variables Bayesian Optimisation will sweep
VARIABLES = ['GTE', 'GTI', 'FRA', 'Pressure']

# Exact bounds from optimizer.py VARIABLE_RANGES
VARIABLE_RANGES = {
    'GTE':      (500,  1100),  # Growth Temperature [C]
    'GTI':      (5,    60),    # Growth Time [min]
    'FRA':      (0,    100),   # Ar Flow Rate [sccm]
    'Pressure': (1,    760),   # Chamber Pressure [Torr]
}

TARGET = 'PL_FWHM'

print(f'Categorical constants ({len(CAT_CONSTANTS)}): {CAT_CONSTANTS}')
print(f'Numerical constants  ({len(NUM_CONSTANTS)}): {NUM_CONSTANTS}')
print(f'\nOptimization variables:')
for v, (lo, hi) in VARIABLE_RANGES.items():
    obs = df[v].dropna().tolist()
    print(f'  {v:12s}  range [{lo} - {hi}]  observed: {obs}')
print(f'\nTarget: {TARGET}  (minimize = better crystal quality)')

## Step 4 — Encode Categorical Constants
Same as `ThermalCVDEncoder.fit_on_data()` — LabelEncoder per categorical column.

In [ ]:
label_encoders = {}
constant_values = {}

print('Categorical encoding:')
for col in CAT_CONSTANTS:
    series = df[col].fillna('Unknown').astype(str)
    le = LabelEncoder()
    le.fit(series.unique())
    label_encoders[col] = le
    constant_values[col] = series.mode()[0]
    print(f'  {col:12s} mode={repr(constant_values[col]):20s} classes={list(le.classes_)}')

print('\nNumerical constant medians:')
for col in NUM_CONSTANTS:
    if col in df.columns:
        val = pd.to_numeric(df[col], errors='coerce').dropna()
        constant_values[col] = float(val.median()) if len(val) > 0 else 0.0
    else:
        constant_values[col] = 0.0
    print(f'  {col:6s} = {constant_values[col]}')

## Step 5 — Build Feature Matrix & Scale
Builds 4-feature matrix `[GTE, GTI, FRA, Pressure]`, imputes missing values, then scales.
Identical to `_build_raw_feature_matrix()` in `data_encoder.py`.

In [ ]:
# Build raw matrix from the 4 optimization variables
X_raw = df[VARIABLES].apply(pd.to_numeric, errors='coerce').values.astype(float)
y_raw = pd.to_numeric(df[TARGET], errors='coerce').values.astype(float)

# Drop rows where target is missing
valid_mask = ~np.isnan(y_raw)
X_raw = X_raw[valid_mask]
y_raw = y_raw[valid_mask]

# Impute missing variable values with column mean (same as app)
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_raw)

# Scale X — StandardScaler, same as app's scaler_X
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_imputed)

# Scale y — StandardScaler, same as app's scaler_y
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y_raw.reshape(-1, 1)).ravel()

print(f'Feature matrix: {X_scaled.shape}  ->  variables={VARIABLES}')
print(f'y (raw meV):  {y_raw.tolist()}')
print(f'y (scaled):   {np.round(y_scaled, 3).tolist()}')

pd.DataFrame(X_scaled, columns=VARIABLES).assign(
    FWHM_raw=y_raw, FWHM_scaled=y_scaled
).style.set_caption('Scaled Feature Matrix').background_gradient(
    subset=['FWHM_raw'], cmap='RdYlGn_r')

## Step 6 — Generate Search Space
5000-point Latin Hypercube over all 4 variable bounds.
Same as `generate_search_space(n_points=5000)` in `optimizer.py`.

In [ ]:
N_SEARCH = 5000

var_names = list(VARIABLE_RANGES.keys())
l_bounds  = [VARIABLE_RANGES[v][0] for v in var_names]
u_bounds  = [VARIABLE_RANGES[v][1] for v in var_names]

sampler      = qmc.LatinHypercube(d=len(var_names), seed=42)
sample       = sampler.random(n=N_SEARCH)
X_search_raw = qmc.scale(sample, l_bounds, u_bounds)

# Apply same imputer + scaler used on training data
X_search = scaler_X.transform(imputer.transform(X_search_raw))

print(f'Search space: {X_search.shape}  (LHS seed=42, n={N_SEARCH})')
pd.DataFrame(X_search_raw[:5], columns=var_names).round(2)

## Step 7 — Train Gaussian Process
Exact kernel from `ThermalCVDGPModel` in `gp_model.py`:
`ConstantKernel * Matern(nu=2.5, ARD) + WhiteKernel`

In [ ]:
# Exact same kernel as gp_model.py
kernel = (
    C(1.0, (0.1, 10.0))
    * Matern(length_scale=[1.0]*4, length_scale_bounds=(0.1, 10.0), nu=2.5)
    + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-4, 1e-1))
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=15,
    normalize_y=True,
    random_state=42
)
gp.fit(X_scaled, y_scaled)

# Predict on training set
y_pred_sc, y_std_sc = gp.predict(X_scaled, return_std=True)
y_pred = scaler_y.inverse_transform(y_pred_sc.reshape(-1, 1)).ravel()
y_std  = y_std_sc * scaler_y.scale_[0]

# Leave-One-Out cross-validation
loo    = LeaveOneOut()
scores = cross_val_score(gp, X_scaled, y_scaled, cv=loo, scoring='r2')

print('GP trained  (ARD Matern nu=2.5 + WhiteKernel)')
print(f'  Optimized kernel: {gp.kernel_}')
print(f'  LOO R2 scores:    {scores.round(3)}')
print(f'  Mean LOO R2:      {scores.mean():.3f}')
print()
print('Predicted vs Actual FWHM:')
for i, (act, pred, std) in enumerate(zip(y_raw, y_pred, y_std)):
    print(f'  Exp {i+1:2d}: actual={act:.0f} meV  predicted={pred:.1f} +/- {std:.1f} meV')

## Step 8 — Expected Improvement & Next Experiment
Same EI formula used by the app optimizer.

In [ ]:
def expected_improvement(X_cand, gp, best_y_scaled, xi=0.01):
    mu, sigma = gp.predict(X_cand, return_std=True)
    sigma = sigma.ravel()
    mu    = mu.ravel()
    z  = (best_y_scaled - mu - xi) / (sigma + 1e-9)
    ei = (best_y_scaled - mu - xi) * norm.cdf(z) + sigma * norm.pdf(z)
    ei[sigma < 1e-10] = 0.0
    return ei

best_idx   = int(np.argmin(y_raw))
best_y_sc  = y_scaled[best_idx]

ei_values   = expected_improvement(X_search, gp, best_y_sc)
best_ei_idx = int(np.argmax(ei_values))
next_raw    = X_search_raw[best_ei_idx]
next_params = dict(zip(var_names, next_raw))

print(f'Current best: Exp {best_idx+1}  FWHM = {y_raw[best_idx]:.0f} meV')
print(f'Max EI: {ei_values.max():.6f}')
print()
print('=== NEXT SUGGESTED EXPERIMENT ===')
for v, val in next_params.items():
    lo, hi = VARIABLE_RANGES[v]
    print(f'  {v:12s}: {val:8.2f}   (range {lo} - {hi})')

## Step 9 — Visualize GP Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Predicted vs Actual ---
ax = axes[0]
ax.errorbar(y_raw, y_pred, yerr=2*y_std, fmt='none',
            color='#AED6F1', alpha=0.7, capsize=5, label='+/-2sigma')
sc = ax.scatter(y_raw, y_pred, c=y_raw, cmap='RdYlGn_r', s=150,
                edgecolors='black', zorder=5,
                vmin=y_raw.min(), vmax=y_raw.max())
plt.colorbar(sc, ax=ax, label='Actual FWHM (meV)')
for i, (xa, xp) in enumerate(zip(y_raw, y_pred)):
    ax.annotate(f'Exp {i+1}', (xa, xp), xytext=(5, 5),
                textcoords='offset points', fontsize=8)
lims = [min(y_raw.min(), y_pred.min())-5, max(y_raw.max(), y_pred.max())+5]
ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.5, label='Perfect fit')
ax.set_xlabel('Actual PL FWHM (meV)')
ax.set_ylabel('GP Predicted PL FWHM (meV)')
ax.set_title('GP: Predicted vs Actual', fontweight='bold')
ax.legend(fontsize=8)

# --- Residuals ---
ax2 = axes[1]
residuals  = y_pred - y_raw
bar_colors = ['#5C9E31' if abs(r) < 10 else '#E84855' for r in residuals]
ax2.bar(range(len(residuals)), residuals, color=bar_colors, edgecolor='white')
ax2.axhline(0, color='black', lw=1.5, linestyle='--')
ax2.set_xticks(range(len(residuals)))
ax2.set_xticklabels([f'Exp {i+1}' for i in range(len(residuals))])
ax2.set_xlabel('Experiment')
ax2.set_ylabel('Residual (meV)')
ax2.set_title('Prediction Residuals', fontweight='bold')

plt.suptitle('Gaussian Process Surrogate Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gp_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 — EI Landscape (2D slices)
Shows where Bayesian Optimization wants to search next.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pairs = [('GTE', 'GTI'), ('FRA', 'Pressure')]

for ax, (vx, vy) in zip(axes, pairs):
    xi_idx = VARIABLES.index(vx)
    yi_idx = VARIABLES.index(vy)
    x_lin  = np.linspace(VARIABLE_RANGES[vx][0], VARIABLE_RANGES[vx][1], 60)
    y_lin  = np.linspace(VARIABLE_RANGES[vy][0], VARIABLE_RANGES[vy][1], 60)
    XX, YY = np.meshgrid(x_lin, y_lin)

    grid_raw = np.tile(imputer.statistics_, (XX.size, 1))
    grid_raw[:, xi_idx] = XX.ravel()
    grid_raw[:, yi_idx] = YY.ravel()
    grid_sc = scaler_X.transform(grid_raw)
    EI_map  = expected_improvement(grid_sc, gp, best_y_sc).reshape(XX.shape)

    cf = ax.contourf(XX, YY, EI_map, levels=25, cmap='YlOrRd')
    plt.colorbar(cf, ax=ax, label='Expected Improvement')
    _obs = df[[vx, vy]].dropna()
    ax.scatter(_obs[vx], _obs[vy],
               c='white', edgecolors='black', s=80, zorder=5, label='Observed')
    ax.scatter([next_params[vx]], [next_params[vy]],
               marker='*', s=300, c='#5C9E31', edgecolors='black',
               zorder=6, label='Next suggestion')
    ax.set_xlabel(vx); ax.set_ylabel(vy)
    ax.set_title(f'EI: {vx} vs {vy}', fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Acquisition Function — Expected Improvement (2D slices)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ei_landscape.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11 — Summary

In [ ]:
print('=' * 60)
print('  WS2 THERMAL CVD — BAYESIAN OPTIMIZATION SUMMARY')
print('=' * 60)
print(f'  Dataset  : {FILE}')
print(f'  Samples  : {len(y_raw)} Thermal CVD rows with measured FWHM')
print(f'  Variables: {VARIABLES}')
print(f'  Target   : {TARGET}')
print()
print(f'  Best FWHM observed : {y_raw[best_idx]:.0f} meV  (Exp {best_idx+1})')
print(f'  Worst FWHM observed: {y_raw.max():.0f} meV')
print(f'  Mean FWHM          : {y_raw.mean():.1f} meV')
print()
print(f'  Kernel  : ARD Matern nu=2.5 + ConstantKernel + WhiteKernel')
print(f'  LOO R2  : {scores.mean():.3f}')
print()
print('  NEXT EXPERIMENT:')
for v, val in next_params.items():
    print(f'    {v:12s}: {val:.2f}')
print('=' * 60)